In [2]:
# 1. Garante que as bibliotecas estão instaladas
!pip install -q openai-whisper ffmpeg-python nltk

import whisper
import collections
import nltk
import heapq # Para selecionar as melhores sentenças
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from google.colab import files

# Downloads necessários
nltk.download('punkt')
nltk.download('stopwords')

# 2. Carregar o modelo Whisper
print("Carregando modelo Whisper...")
modelo = whisper.load_model("medium")

# 3. Janela de Seleção de Arquivo
print("\nSelecione o arquivo de áudio no seu computador:")
uploaded = files.upload()

if uploaded:
    nome_arquivo = list(uploaded.keys())[0]

    try:
        print(f"\nProcessando transcrição: {nome_arquivo}...")
        resultado = modelo.transcribe(nome_arquivo, fp16=False)
        texto_final = resultado["text"]

        print("\n" + "="*40)
        print("TRANSCRIÇÃO COMPLETA:")
        print(texto_final)
        print("="*40)

        # --- 4. LÓGICA DE RESUMO ---
        print("\nGerando resumo...")

        # Tokenização e Stopwords
        sentencas = sent_tokenize(texto_final)
        stop_words = set(stopwords.words('portuguese'))

        # Calcula a frequência das palavras
        frequencia_palavras = collections.Counter(
            [w.lower() for w in word_tokenize(texto_final) if w.isalpha() and w.lower() not in stop_words]
        )

        # Normaliza a frequência (0 a 1)
        max_freq = max(frequencia_palavras.values()) if frequencia_palavras else 1
        for palavra in frequencia_palavras.keys():
            frequencia_palavras[palavra] /= max_freq

        # Pontua as sentenças baseada nas palavras frequentes
        notas_sentencas = {}
        for sentenca in sentencas:
            for palavra in word_tokenize(sentenca.lower()):
                if palavra in frequencia_palavras:
                    if sentenca not in notas_sentencas:
                        notas_sentencas[sentenca] = frequencia_palavras[palavra]
                    else:
                        notas_sentencas[sentenca] += frequencia_palavras[palavra]

        # Seleciona as 3 frases mais importantes (você pode ajustar esse número)
        resumo_lista = heapq.nlargest(3, notas_sentencas, key=notas_sentencas.get)
        resumo = " ".join(resumo_lista)

        print("\n" + "•"*40)
        print("RESUMO DO ÁUDIO:")
        print(resumo if resumo else "O áudio é curto demais para um resumo.")
        print("•"*40)

        # 5. Análise de palavras-chave
        limpas = [w.lower() for w in word_tokenize(texto_final) if w.isalpha() and w.lower() not in stop_words]
        print("\nTOP 5 PALAVRAS-CHAVE:")
        for pal, freq in collections.Counter(limpas).most_common(5):
            print(f"- {pal.capitalize()}: {freq}x")

    except Exception as e:
        print(f"Erro ao processar: {e}")
else:
    print("Nenhum arquivo foi selecionado.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Carregando modelo Whisper...

Selecione o arquivo de áudio no seu computador:


Saving texto_convertido (1).mp3 to texto_convertido (1) (1).mp3

Processando transcrição: texto_convertido (1) (1).mp3...

TRANSCRIÇÃO COMPLETA:
 Então, galera, a gente está, tipo, começando a estruturar aquela nova funcionalidade do sistema, né? Aí, eu estava pensando aqui que, meio que a gente precisa validar os dados antes de subir para o Firebase. Tá, por quê? Então, se a gente só jogar lá, pode dar erro no dashboard de BI? Tipo, o gestor vai ver os indicadores errados, né? Aí complica o nosso lado. Então, a ideia é, tipo, criar uma trava no código, tá ligado? Meio que para garantir que o usuário não faça besteira, né? Então, basicamente é isso. Tá, aí qualquer dúvida vocês me avisam, né? Porque a gente precisa entregar isso rápido. Então, vamos focar nisso aí. Tá, tchau, tchau, tchau!

Gerando resumo...

••••••••••••••••••••••••••••••••••••••••
RESUMO DO ÁUDIO:
 Então, galera, a gente está, tipo, começando a estruturar aquela nova funcionalidade do sistema, né? Aí, eu estava pensa